In [5]:
from collections import deque

# 1. Clase base del problema
class Problem:
    def __init__(self, initial, goal):
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        raise NotImplementedError

    def result(self, state, action):
        raise NotImplementedError

    def is_goal(self, state):
        return self.goal == state

    def action_cost(self, state1, action, state2):
        return 1

    def h(self, state):
        return 0

# 2. Problema adaptado a Grafos
class GraphProblem(Problem):
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph

    def actions(self, state):
        lista = []
        # Usa .get() para prevenir KeyError
        for key in self.graph.get(state, {}).keys():
            lista.append(key)
        return lista

    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        return self.graph[state1][state2]

# 3. Clase Nodo para manejar la frontera y los caminos
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def path(self):
        lista_path = []
        node = self
        while node:
            lista_path.append(node.state)
            node = node.parent
        return lista_path[::-1]

    def expand(self, problem):
        lista = []
        for action in problem.actions(self.state):
            lista.append(self.child_node(problem, action))
        return lista

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)
        return Node(next_state, self, action, self.path_cost + step_cost)

In [6]:
# DFS
def depth_first_graph_search(problem):
    start_node = Node(problem.initial)
    if problem.is_goal(start_node.state):
        return start_node
    frontier = [start_node]
    explored = set()

    while frontier:
        node = frontier.pop()
        explored.add(node.state)

        if problem.is_goal(node.state):
            return node

        for child in node.expand(problem):
            if child.state not in explored and child not in frontier:
                frontier.append(child)
    return None

# BFS
def breadth_first_graph_search(problem):
    start_node = Node(problem.initial)
    if problem.is_goal(start_node.state):
        return start_node
    frontier = deque([start_node])
    explored = set()

    while frontier:
        node = frontier.popleft()
        explored.add(node.state)

        if problem.is_goal(node.state):
            return node

        for child in node.expand(problem):
            if child.state not in explored and child not in frontier:
                frontier.append(child)
    return None

In [7]:
metro_cdmx = {
    # Línea 1
    'Pantitlan': {'Zaragoza': 1, 'Oceania': 1, 'Puebla': 1, 'Hangares': 1},
    'Zaragoza': {'Pantitlan': 1, 'Gomez Farias': 1},
    'Gomez Farias': {'Zaragoza': 1, 'Boulevard Puerto Aereo': 1},
    'Boulevard Puerto Aereo': {'Gomez Farias': 1, 'Balbuena': 1},
    'Balbuena': {'Boulevard Puerto Aereo': 1, 'Moctezuma': 1},
    'Moctezuma': {'Balbuena': 1, 'San Lazaro': 1},
    'San Lazaro': {'Moctezuma': 1, 'Candelaria': 1, 'Morelos': 1},
    'Candelaria': {'San Lazaro': 1, 'Merced': 1, 'Fray Servando': 1, 'Morelos': 1},
    'Merced': {'Candelaria': 1, 'Pino Suarez': 1},
    'Pino Suarez': {'Merced': 1, 'Isabel la Catolica': 1, 'San Antonio Abad': 1, 'Zocalo': 1},
    'Isabel la Catolica': {'Pino Suarez': 1, 'Salto del Agua': 1},
    'Salto del Agua': {'Isabel la Catolica': 1, 'Balderas': 1, 'Bellas Artes': 1, 'Doctores': 1},
    'Balderas': {'Salto del Agua': 1, 'Cuauhtemoc': 1, 'Juarez': 1, 'Niños Heroes': 1},
    'Cuauhtemoc': {'Balderas': 1, 'Insurgentes': 1},
    'Insurgentes': {'Cuauhtemoc': 1, 'Sevilla': 1},
    'Sevilla': {'Insurgentes': 1, 'Chapultepec': 1},
    'Chapultepec': {'Sevilla': 1, 'Juanacatlan': 1},
    'Juanacatlan': {'Chapultepec': 1, 'Tacubaya': 1},
    'Tacubaya': {'Juanacatlan': 1, 'Observatorio': 1, 'Constituyentes': 1, 'San Pedro de los Pinos': 1},
    'Observatorio': {'Tacubaya': 1},

    # Línea 2
    'Cuatro Caminos': {'Panteones': 1},
    'Panteones': {'Cuatro Caminos': 1, 'Tacuba': 1},
    'Tacuba': {'Panteones': 1, 'Cuitlahuac': 1, 'Refineria': 1, 'Popotla': 1},
    'Cuitlahuac': {'Tacuba': 1, 'Popotla': 1},
    'Popotla': {'Cuitlahuac': 1, 'Colegio Militar': 1},
    'Colegio Militar': {'Popotla': 1, 'Normal': 1},
    'Normal': {'Colegio Militar': 1, 'San Cosme': 1},
    'San Cosme': {'Normal': 1, 'Revolucion': 1},
    'Revolucion': {'San Cosme': 1, 'Hidalgo': 1},
    'Hidalgo': {'Revolucion': 1, 'Bellas Artes': 1, 'Guerrero': 1, 'Juarez': 1},
    'Bellas Artes': {'Hidalgo': 1, 'Allende': 1, 'Salto del Agua': 1, 'Garibaldi': 1},
    'Allende': {'Bellas Artes': 1, 'Zocalo': 1},
    'Zocalo': {'Allende': 1, 'Pino Suarez': 1},
    'San Antonio Abad': {'Pino Suarez': 1, 'Chabacano': 1},
    'Chabacano': {'San Antonio Abad': 1, 'Viaducto': 1, 'Lázaro Cárdenas': 1, 'Jamaica': 1, 'Obrero Mundial': 1},
    'Viaducto': {'Chabacano': 1, 'Xola': 1},
    'Xola': {'Viaducto': 1, 'Villa de Cortes': 1},
    'Villa de Cortes': {'Xola': 1, 'Nativitas': 1},
    'Nativitas': {'Villa de Cortes': 1, 'Portales': 1},
    'Portales': {'Nativitas': 1, 'Ermita': 1},
    'Ermita': {'Portales': 1, 'General Anaya': 1, 'Eje Central': 1, 'Mexicaltzingo': 1},
    'General Anaya': {'Ermita': 1, 'Taxqueña': 1},
    'Taxqueña': {'General Anaya': 1},

    # Línea 3
    'Politécnico': {'Instituto del Petroleo': 1},
    'Instituto del Petroleo': {'Politécnico': 1, 'Lindavista': 1, 'Autobuses del Norte': 1, 'Ferreria': 1},
    'Autobuses del Norte': {'Instituto del Petroleo': 1, 'La Raza': 1},
    'La Raza': {'Autobuses del Norte': 1, 'Potrero': 1, 'Misterios': 1, 'Tlatelolco': 1},
    'Potrero': {'La Raza': 1, 'Deportivo 18 de Marzo': 1},
    'Deportivo 18 de Marzo': {'Potrero': 1, 'Indios Verdes': 1, 'Lindavista': 1},
    'Indios Verdes': {'Deportivo 18 de Marzo': 1},
    'Tlatelolco': {'La Raza': 1, 'Guerrero': 1},
    'Guerrero': {'Tlatelolco': 1, 'Hidalgo': 1, 'Buenavista': 1},
    'Juarez': {'Hidalgo': 1, 'Balderas': 1},
    'Niños Heroes': {'Balderas': 1, 'Hospital General': 1},
    'Hospital General': {'Niños Heroes': 1, 'Centro Medico': 1},
    'Centro Medico': {'Hospital General': 1, 'Etiopia': 1, 'Lázaro Cárdenas': 1, 'Chilpancingo': 1},
    'Chilpancingo': {'Centro Medico': 1, 'Patriotismo': 1},
    'Patriotismo': {'Chilpancingo': 1, 'Tacubaya': 1},
    'Etiopia': {'Centro Medico': 1, 'Eugenia': 1},
    'Eugenia': {'Etiopia': 1, 'Division del Norte': 1},
    'Division del Norte': {'Eugenia': 1, 'Zapata': 1},
    'Zapata': {'Division del Norte': 1, 'Coyoacan': 1, 'Parque de los Venados': 1, 'Hospital 20 de Noviembre': 1},
    'Coyoacan': {'Zapata': 1, 'Viveros': 1},
    'Viveros': {'Coyoacan': 1, 'Miguel Angel de Quevedo': 1},
    'Miguel Angel de Quevedo': {'Viveros': 1, 'Copilco': 1},
    'Copilco': {'Miguel Angel de Quevedo': 1, 'Universidad': 1},
    'Universidad': {'Copilco': 1},

    # Estaciones adicionales
    'Oceania': {'Pantitlan': 1, 'Terminal Aerea': 1, 'Aragon': 1, 'Romero Rubio': 1},
    'Terminal Aerea': {'Oceania': 1, 'Hangares': 1},
    'Hangares': {'Terminal Aerea': 1, 'Pantitlan': 1},
    'Puebla': {'Pantitlan': 1},
    'Morelos': {'San Lazaro': 1, 'Candelaria': 1},
    'Fray Servando': {'Candelaria': 1},
    'Doctores': {'Salto del Agua': 1},
    'Constituyentes': {'Tacubaya': 1},
    'San Pedro de los Pinos': {'Tacubaya': 1},
    'Refineria': {'Tacuba': 1},
    'Garibaldi': {'Bellas Artes': 1},
    'Lázaro Cárdenas': {'Chabacano': 1, 'Centro Medico': 1},
    'Jamaica': {'Chabacano': 1},
    'Obrero Mundial': {'Chabacano': 1},
    'Eje Central': {'Ermita': 1},
    'Mexicaltzingo': {'Ermita': 1},
    'Lindavista': {'Instituto del Petroleo': 1, 'Deportivo 18 de Marzo': 1},
    'Ferreria': {'Instituto del Petroleo': 1},
    'Misterios': {'La Raza': 1},
    'Buenavista': {'Guerrero': 1},
    'Parque de los Venados': {'Zapata': 1},
    'Hospital 20 de Noviembre': {'Zapata': 1},
    'Aragon': {'Oceania': 1},
    'Romero Rubio': {'Oceania': 1}
}

In [8]:
rutas = [
    ("Cuatro Caminos", "Pantitlan"),
    ("Politécnico", "Taxqueña"),
    ("Zapata", "Oceania")
]

for origen, destino in rutas:
    print(f"RUTA SOLICITADA: {origen} -> {destino}")

    
    # Resolviendo con DFS
    prob_dfs = GraphProblem(origen, destino, metro_cdmx)
    nodo_dfs = depth_first_graph_search(prob_dfs)
    path_dfs = nodo_dfs.path() if nodo_dfs else []
    
    # Resolviendo con BFS
    prob_bfs = GraphProblem(origen, destino, metro_cdmx)
    nodo_bfs = breadth_first_graph_search(prob_bfs)
    path_bfs = nodo_bfs.path() if nodo_bfs else []
    
    print(f"• DFS (Costo: {nodo_dfs.path_cost if nodo_dfs else 0}):")
    print(path_dfs)
    print(f"\n• BFS (Costo óptimo: {nodo_bfs.path_cost if nodo_bfs else 0}):")
    print(path_bfs)
    print("\n")

RUTA SOLICITADA: Cuatro Caminos -> Pantitlan
• DFS (Costo: 27):
['Cuatro Caminos', 'Panteones', 'Tacuba', 'Popotla', 'Colegio Militar', 'Normal', 'San Cosme', 'Revolucion', 'Hidalgo', 'Juarez', 'Balderas', 'Niños Heroes', 'Hospital General', 'Centro Medico', 'Lázaro Cárdenas', 'Chabacano', 'San Antonio Abad', 'Pino Suarez', 'Merced', 'Candelaria', 'Morelos', 'San Lazaro', 'Moctezuma', 'Balbuena', 'Boulevard Puerto Aereo', 'Gomez Farias', 'Zaragoza', 'Pantitlan']

• BFS (Costo óptimo: 21):
['Cuatro Caminos', 'Panteones', 'Tacuba', 'Popotla', 'Colegio Militar', 'Normal', 'San Cosme', 'Revolucion', 'Hidalgo', 'Bellas Artes', 'Allende', 'Zocalo', 'Pino Suarez', 'Merced', 'Candelaria', 'San Lazaro', 'Moctezuma', 'Balbuena', 'Boulevard Puerto Aereo', 'Gomez Farias', 'Zaragoza', 'Pantitlan']


RUTA SOLICITADA: Politécnico -> Taxqueña
• DFS (Costo: 21):
['Politécnico', 'Instituto del Petroleo', 'Autobuses del Norte', 'La Raza', 'Tlatelolco', 'Guerrero', 'Hidalgo', 'Juarez', 'Balderas', 'Niños 